# Lesson 3a: Monte Carlo Methods - Theory

**Reinforcement Learning Course - Powell Clark**

---

## Learning Objectives

By the end of this notebook, you will:

1. **Understand Model-Free RL** - Learning without knowing transition dynamics
2. **Master Monte Carlo Prediction** - Estimating value functions from experience
3. **Learn MC Control** - Finding optimal policies through episodic interaction
4. **Explore Exploration Strategies** - Epsilon-greedy and exploring starts
5. **Compare First-Visit vs Every-Visit** - Different MC variants
6. **Understand On-Policy vs Off-Policy** - Two fundamental approaches
7. **Derive Incremental Updates** - Efficient online learning

This lesson marks the transition from dynamic programming (which requires a model) to model-free methods that learn directly from experience.

---

## Table of Contents

1. [Introduction to Model-Free RL](#intro)
2. [Monte Carlo Prediction](#prediction)
3. [Monte Carlo Control](#control)
4. [Exploration vs Exploitation](#exploration)
5. [First-Visit vs Every-Visit](#variants)
6. [Incremental Implementation](#incremental)
7. [On-Policy vs Off-Policy](#on-off)
8. [Theoretical Guarantees](#theory)
9. [Advantages and Limitations](#limits)

---

## Setup

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from typing import List, Tuple, Dict
from collections import defaultdict

plt.style.use('seaborn-v0_8-darkgrid')
sns.set_palette("husl")
%matplotlib inline

np.random.seed(42)
print("✅ Setup complete")

## 1. Introduction to Model-Free RL <a id='intro'></a>

### The Fundamental Challenge

In dynamic programming, we assumed full knowledge of the MDP:
- Transition probabilities: $p(s'|s,a)$
- Reward function: $r(s,a,s')$

**Reality**: In most real-world problems, we don't have this model!

### What is Model-Free RL?

**Model-free methods** learn value functions and policies directly from experience, without building an explicit model of the environment.

**Experience** = sequences of $(S_t, A_t, R_{t+1}, S_{t+1})$ tuples

### Why Monte Carlo?

**Monte Carlo (MC) methods** use the simplest possible idea:

> *"To learn the value of a state, simply average the returns obtained after visiting that state."*

**Return** from time $t$:
$$G_t = R_{t+1} + \gamma R_{t+2} + \gamma^2 R_{t+3} + ... + \gamma^{T-t-1} R_T$$

By definition of value function:
$$V^\pi(s) = \mathbb{E}_\pi[G_t | S_t = s]$$

**MC Estimation**:
$$V(s) \approx \text{average of observed returns from } s$$

### Key Requirements

Monte Carlo methods require:
1. **Episodic tasks**: Episodes must terminate
2. **Complete episodes**: Must observe full return $G_t$
3. **No model needed**: Learn from raw experience

### Comparison: DP vs MC

| Aspect | Dynamic Programming | Monte Carlo |
|--------|-------------------|-------------|
| **Model** | Required | Not required |
| **Updates** | Bootstrap (use estimates) | Use actual returns |
| **Episodes** | Not required | Required (episodic) |
| **Convergence** | Guaranteed (finite MDP) | Guaranteed (with conditions) |
| **Computation** | Full sweeps | Sample-based |
| **Bias** | None (exact Bellman) | None (sample averages) |
| **Variance** | Low | High (stochastic returns) |

## 2. Monte Carlo Prediction <a id='prediction'></a>

### Problem Statement

**Goal**: Estimate $V^\pi(s)$ for all states under a given policy $\pi$

**Given**: Ability to generate episodes following $\pi$

### First-Visit MC Prediction

**Algorithm**:

```
Input: policy π to evaluate
Initialize:
  V(s) = 0, for all s ∈ S
  Returns(s) = empty list, for all s ∈ S

Repeat forever:
  Generate episode following π: S₀, A₀, R₁, S₁, A₁, R₂, ..., S_T
  G ← 0
  
  For t = T-1, T-2, ..., 0:
    G ← γG + R_{t+1}
    
    If S_t does not appear in S₀, S₁, ..., S_{t-1}:  # First visit
      Append G to Returns(S_t)
      V(S_t) ← average(Returns(S_t))
```

### Mathematical Foundation

**Law of Large Numbers**:

As the number of visits to state $s$ approaches infinity, the sample mean converges to the expected value:

$$\lim_{n \to \infty} \frac{1}{n} \sum_{i=1}^n G_i^{(s)} = \mathbb{E}_\pi[G_t | S_t = s] = V^\pi(s)$$

where $G_i^{(s)}$ is the return from the $i$-th visit to state $s$.

### Every-Visit MC Prediction

**Difference**: Average returns from *all* visits to a state, not just first visits.

**Algorithm modification**:
```
For t = T-1, T-2, ..., 0:
  G ← γG + R_{t+1}
  Append G to Returns(S_t)      # No first-visit check
  V(S_t) ← average(Returns(S_t))
```

**Both methods converge to $V^\pi(s)$**, though:
- First-visit: Estimates are unbiased
- Every-visit: May have bias but often lower variance

### Example: Blackjack

Consider estimating value of having 20 in Blackjack:

**Episode 1**: Start with 20 → Stand → Dealer busts → Win (+1)
- $G_1 = +1$
- $V(20) = +1$

**Episode 2**: Start with 20 → Stand → Dealer gets 21 → Lose (-1)
- $G_2 = -1$
- $V(20) = \frac{1 + (-1)}{2} = 0$

**Episode 3**: Start with 20 → Stand → Dealer gets 19 → Win (+1)
- $G_3 = +1$
- $V(20) = \frac{1 + (-1) + 1}{3} = 0.33$

Continue averaging returns...

## 3. Monte Carlo Control <a id='control'></a>

### Problem Statement

**Goal**: Find optimal policy $\pi^*$ without a model

**Approach**: Generalized Policy Iteration (GPI) with MC

### Why We Need Q-Values

With a model, we could improve policy using $V$:
$$\pi'(s) = \arg\max_a \sum_{s'} p(s'|s,a)[r + \gamma V(s')]$$

**Without a model**, we can't compute this! We need action values:
$$Q^\pi(s,a) = \mathbb{E}_\pi[G_t | S_t = s, A_t = a]$$

Then policy improvement becomes:
$$\pi'(s) = \arg\max_a Q(s,a)$$

### Monte Carlo ES (Exploring Starts)

**Exploring Starts**: Every state-action pair has non-zero probability of being selected as the start.

**Algorithm**:

```
Initialize:
  Q(s,a) arbitrarily, for all s ∈ S, a ∈ A
  Returns(s,a) ← empty list, for all s, a
  π(s) ← arbitrary deterministic policy

Repeat forever:
  # Generate episode with exploring starts
  Choose S₀ ∈ S, A₀ ∈ A randomly (all pairs have probability > 0)
  Generate episode starting from S₀, A₀ following π:
    S₀, A₀, R₁, S₁, A₁, R₂, ..., S_T
  
  G ← 0
  For t = T-1, T-2, ..., 0:
    G ← γG + R_{t+1}
    
    If (S_t, A_t) does not appear in (S₀,A₀), ..., (S_{t-1}, A_{t-1}):
      Append G to Returns(S_t, A_t)
      Q(S_t, A_t) ← average(Returns(S_t, A_t))
      π(S_t) ← argmax_a Q(S_t, a)
```

### Convergence of MC ES

**Theorem 3.1**: MC ES converges to an optimal policy.

**Proof sketch**:

1. **Policy evaluation**: $Q$ estimates converge to $Q^\pi$ (law of large numbers)

2. **Policy improvement**: Greedy policy with respect to $Q^\pi$ is better than or equal to $\pi$ (policy improvement theorem)

3. **Monotonic improvement**: Each policy iteration improves or maintains performance

4. **Finite convergence**: Finite number of policies → must converge to optimal

$\square$

### The Exploration Problem

**Issue**: Exploring starts may not be practical:
- Can't control initial state in real environments
- May be dangerous (e.g., robotic systems)

**Solution**: Ensure exploration through the policy itself!

## 4. Exploration vs Exploitation <a id='exploration'></a>

### The Exploration-Exploitation Dilemma

**Exploitation**: Choose actions that maximize immediate reward based on current knowledge

**Exploration**: Try different actions to discover potentially better options

**Dilemma**: Can't do both simultaneously, but need both for optimal learning!

### Epsilon-Greedy Policies

**Definition**: An $\epsilon$-greedy policy with respect to $Q$:

$$\pi(a|s) = \begin{cases}
1 - \epsilon + \frac{\epsilon}{|\mathcal{A}(s)|} & \text{if } a = \arg\max_{a'} Q(s,a') \\
\frac{\epsilon}{|\mathcal{A}(s)|} & \text{otherwise}
\end{cases}$$

**Interpretation**:
- With probability $1 - \epsilon$: Choose greedy action (exploit)
- With probability $\epsilon$: Choose random action (explore)

### Properties of $\epsilon$-Greedy

1. **Guaranteed exploration**: All actions tried infinitely often
2. **Simple to implement**: Single parameter $\epsilon$
3. **Bounded suboptimality**: Performance gap decreases as $\epsilon \to 0$
4. **Flexible**: Can decrease $\epsilon$ over time (annealing)

### GLIE (Greedy in the Limit with Infinite Exploration)

**Definition**: A policy sequence is GLIE if:

1. **All state-action pairs are visited infinitely often**:
   $$\lim_{k \to \infty} N_k(s,a) = \infty \quad \forall s,a$$

2. **Policy converges to greedy policy**:
   $$\lim_{k \to \infty} \pi_k(a|s) = \mathbb{1}(a = \arg\max_{a'} Q_k(s,a'))$$

**Example**: $\epsilon$-greedy with $\epsilon_k = \frac{1}{k}$ is GLIE

### Exploration Schedules

**Constant $\epsilon$**:
- $\epsilon(t) = 0.1$
- Pro: Continuous exploration
- Con: Never fully exploits

**Linear decay**:
- $\epsilon(t) = \max(\epsilon_{\min}, \epsilon_0 - \frac{t}{T})$
- Pro: Smooth transition
- Con: Requires tuning $T$

**Exponential decay**:
- $\epsilon(t) = \epsilon_0 e^{-\lambda t}$
- Pro: Fast initial exploration, slow convergence
- Con: May decay too quickly

**Inverse decay** (GLIE):
- $\epsilon(t) = \frac{1}{t}$ or $\frac{c}{c + t}$
- Pro: Theoretical guarantees
- Con: Very slow decay

## 5. First-Visit vs Every-Visit <a id='variants'></a>

### Detailed Comparison

#### First-Visit MC

**Definition**: Average returns only from first visit to each state in an episode

**Advantages**:
- Estimates are **unbiased**: $\mathbb{E}[\hat{V}(s)] = V^\pi(s)$
- Easier to analyze theoretically
- Each episode provides independent samples

**Disadvantages**:
- Discards information from later visits
- May have higher variance
- Requires tracking which states already visited

#### Every-Visit MC

**Definition**: Average returns from all visits to each state

**Advantages**:
- Uses all available data
- Often lower variance in practice
- Simpler implementation (no visit tracking)
- Faster convergence in some problems

**Disadvantages**:
- Estimates may be biased (returns within episode are correlated)
- Bias vanishes asymptotically
- Less studied theoretically

### Convergence Comparison

**Theorem 5.1** (First-Visit MC): 
$$\lim_{n \to \infty} \hat{V}_n(s) = V^\pi(s) \quad \text{almost surely}$$
where $n$ is the number of *first visits* to $s$.

**Theorem 5.2** (Every-Visit MC):
$$\lim_{n \to \infty} \hat{V}_n(s) = V^\pi(s) \quad \text{almost surely}$$
where $n$ is the number of *all visits* to $s$.

Both converge, but at different rates depending on:
- State revisit frequency
- Episode length
- Correlation structure

### When to Use Which?

**Use First-Visit when**:
- Theoretical guarantees are important
- States are rarely revisited in episodes
- Unbiased estimates are critical

**Use Every-Visit when**:
- Data efficiency is important
- States are frequently revisited
- Faster convergence needed
- Implementation simplicity matters

## 6. Incremental Implementation <a id='incremental'></a>

### The Memory Problem

Storing all returns for averaging requires unbounded memory:
```python
Returns(s).append(G)
V(s) = average(Returns(s))  # Stores all returns!
```

### Incremental Mean Update

**Key insight**: Can update mean without storing all values.

**Derivation**:

Let $\mu_n$ be the mean of the first $n$ values $x_1, ..., x_n$:

$$\mu_n = \frac{1}{n} \sum_{i=1}^n x_i$$

For the next value $x_{n+1}$:

$$\mu_{n+1} = \frac{1}{n+1} \sum_{i=1}^{n+1} x_i$$

$$= \frac{1}{n+1} \left( x_{n+1} + \sum_{i=1}^n x_i \right)$$

$$= \frac{1}{n+1} \left( x_{n+1} + n\mu_n \right)$$

$$= \frac{1}{n+1} x_{n+1} + \frac{n}{n+1} \mu_n$$

$$= \mu_n + \frac{1}{n+1}(x_{n+1} - \mu_n)$$

**Incremental update rule**:
$$\text{NewEstimate} \leftarrow \text{OldEstimate} + \frac{1}{n}[\text{Target} - \text{OldEstimate}]$$

### Incremental MC Updates

For value function:
$$V(S_t) \leftarrow V(S_t) + \alpha[G_t - V(S_t)]$$

For action-value function:
$$Q(S_t, A_t) \leftarrow Q(S_t, A_t) + \alpha[G_t - Q(S_t, A_t)]$$

where:
- $\alpha = \frac{1}{n}$ for sample average (decreasing)
- $\alpha = \text{const}$ for exponential recency-weighted average (constant)

### Constant Step-Size

Using constant $\alpha \in (0,1]$:

$$Q_{n+1} = Q_n + \alpha(G_n - Q_n)$$

Expanding:
$$Q_{n+1} = (1-\alpha)Q_n + \alpha G_n$$

$$= (1-\alpha)[(1-\alpha)Q_{n-1} + \alpha G_{n-1}] + \alpha G_n$$

$$= (1-\alpha)^2 Q_{n-1} + \alpha[(1-\alpha)G_{n-1} + G_n]$$

Continuing:
$$Q_{n+1} = (1-\alpha)^n Q_1 + \sum_{i=1}^n \alpha(1-\alpha)^{n-i} G_i$$

**Interpretation**: Exponentially weighted average of returns, with recent returns weighted more heavily.

**Advantages**:
- Adapts to non-stationary environments
- Bounded memory
- Weights recent experience more

**Disadvantages**:
- Biased estimator
- May not converge to true value
- Requires tuning $\alpha$

## 7. On-Policy vs Off-Policy <a id='on-off'></a>

### Two Fundamental Paradigms

#### On-Policy Learning

**Definition**: Evaluate and improve the **same policy** that generates behavior.

- **Behavior policy**: $\pi$ (the policy being evaluated)
- **Target policy**: $\pi$ (same policy)

**Example**: $\epsilon$-greedy Monte Carlo
- Follow $\epsilon$-greedy policy to generate episodes
- Evaluate and improve the $\epsilon$-greedy policy

**Advantages**:
- Simpler conceptually
- Easier to implement
- Natural exploration-exploitation balance
- Typically faster convergence

**Disadvantages**:
- Can only learn near-optimal policies (due to exploration)
- Cannot learn from other agents or old policies

#### Off-Policy Learning

**Definition**: Evaluate/improve one policy while following a **different** policy.

- **Behavior policy**: $b$ (generates experience)
- **Target policy**: $\pi$ (being evaluated/improved)

**Example**: Learn optimal policy while following exploratory policy
- Behavior $b$: Random or $\epsilon$-greedy
- Target $\pi$: Deterministic greedy policy

**Advantages**:
- Learn optimal policy while exploring
- Learn from human demonstrations
- Reuse old experience
- Greater flexibility

**Disadvantages**:
- More complex (importance sampling)
- Higher variance
- Slower convergence
- Requires coverage: $\pi(a|s) > 0 \Rightarrow b(a|s) > 0$

### On-Policy MC Control

**Algorithm** ($\epsilon$-greedy):

```
Initialize Q(s,a) arbitrarily
Initialize ε-greedy policy π from Q

Repeat forever:
  Generate episode using π:
    S₀, A₀, R₁, S₁, A₁, R₂, ..., S_T
  
  G ← 0
  For t = T-1 to 0:
    G ← γG + R_{t+1}
    Q(S_t, A_t) ← Q(S_t, A_t) + α[G - Q(S_t, A_t)]
    π ← ε-greedy w.r.t. Q
```

### Off-Policy MC with Importance Sampling

**Problem**: Returns generated by $b$ have wrong distribution for evaluating $\pi$.

**Solution**: Weight returns by importance sampling ratio.

**Importance Sampling Ratio**:

For trajectory $\tau = (S_t, A_t, R_{t+1}, ..., S_T)$:

$$\rho_{t:T-1} = \prod_{k=t}^{T-1} \frac{\pi(A_k|S_k)}{b(A_k|S_k)}$$

**Weighted Return**: $\rho_{t:T-1} G_t$

**Ordinary Importance Sampling**:
$$V(s) = \frac{\sum_{t \in \mathcal{T}(s)} \rho_{t:T(t)-1} G_t}{|\mathcal{T}(s)|}$$

**Weighted Importance Sampling**:
$$V(s) = \frac{\sum_{t \in \mathcal{T}(s)} \rho_{t:T(t)-1} G_t}{\sum_{t \in \mathcal{T}(s)} \rho_{t:T(t)-1}}$$

**Weighted IS properties**:
- Biased (but bias → 0 as $n → \infty$)
- Lower variance than ordinary IS
- Preferred in practice

## 8. Theoretical Guarantees <a id='theory'></a>

### Convergence of MC Prediction

**Theorem 8.1** (MC Prediction Convergence):

Given policy $\pi$ and state $s$, if state $s$ is visited infinitely often, then the Monte Carlo estimate $\hat{V}(s)$ converges to $V^\pi(s)$ almost surely.

**Proof**: Direct application of the strong law of large numbers to the sample mean of returns. $\square$

### Convergence of On-Policy MC Control

**Theorem 8.2** (GLIE MC Control):

If the policy sequence is GLIE and step-sizes satisfy:
1. $\sum_{t=1}^\infty \alpha_t = \infty$
2. $\sum_{t=1}^\infty \alpha_t^2 < \infty$

Then $Q_t(s,a) \to Q^*(s,a)$ almost surely.

**Proof sketch**:
1. GLIE ensures all state-action pairs visited infinitely
2. Step-size conditions ensure convergence (Robbins-Monro)
3. Policy converges to greedy policy w.r.t. $Q^*$
$\square$

**Example step-sizes satisfying conditions**:
- $\alpha_t = \frac{1}{t}$ ✅
- $\alpha_t = \frac{1}{\sqrt{t}}$ ❌ (doesn't satisfy first condition)
- $\alpha_t = 0.01$ ❌ (doesn't satisfy second condition)

### Policy Improvement for $\epsilon$-Greedy

**Theorem 8.3**: An $\epsilon$-greedy policy $\pi'$ w.r.t. $Q^\pi$ is an improvement over any $\epsilon$-soft policy $\pi$:

$$V^{\pi'}(s) \geq V^\pi(s) \quad \forall s$$

**Proof**:

For any $\epsilon$-soft policy $\pi$:

$$Q^\pi(s, \pi'(s)) = \sum_a \pi'(a|s) Q^\pi(s,a)$$

$$= \frac{\epsilon}{|\mathcal{A}(s)|} \sum_a Q^\pi(s,a) + (1-\epsilon) \max_a Q^\pi(s,a)$$

$$\geq \frac{\epsilon}{|\mathcal{A}(s)|} \sum_a Q^\pi(s,a) + (1-\epsilon) \sum_a \frac{\pi(a|s) - \epsilon/|\mathcal{A}(s)|}{1-\epsilon} Q^\pi(s,a)$$

$$= \sum_a \pi(a|s) Q^\pi(s,a) = V^\pi(s)$$

By policy improvement theorem: $V^{\pi'}(s) \geq V^\pi(s)$ $\square$

### Optimality of $\epsilon$-Greedy Policy

**Important**: $\epsilon$-greedy policies can only converge to **near-optimal** policies, not truly optimal, because they maintain exploration.

**Best achievable with fixed $\epsilon$**:
$$V^{\pi_{\epsilon}} \leq V^* + \mathcal{O}(\epsilon)$$

**Solution**: Use GLIE schedule where $\epsilon \to 0$

## 9. Advantages and Limitations <a id='limits'></a>

### Advantages of Monte Carlo Methods

1. **Model-Free**
   - No need for transition probabilities
   - Learn directly from experience
   - Applicable to real-world problems

2. **Conceptually Simple**
   - Based on averaging returns
   - Easy to understand and implement
   - No bootstrapping (using estimates to update estimates)

3. **Unbiased Estimates**
   - First-visit MC is unbiased
   - No bias from bootstrapping
   - Converges to true values

4. **State Value Independence**
   - Value of one state doesn't depend on others
   - Can focus on subset of states
   - Useful for sample-based planning

5. **Works with Simulation**
   - Generate episodes from simulator
   - No need for explicit MDP model
   - Handles complex environments

### Limitations of Monte Carlo Methods

1. **Requires Episodic Tasks**
   - Must wait for episode to terminate
   - Cannot handle continuing tasks directly
   - Long episodes → slow learning

2. **High Variance**
   - Returns are stochastic
   - Variance from full trajectory
   - Slow convergence in practice
   - Requires many episodes

3. **Delayed Learning**
   - No learning during episode
   - Updates only at episode end
   - Inefficient use of experience

4. **Exploration Challenges**
   - Need to ensure all states visited
   - Exploring starts may be impractical
   - $\epsilon$-greedy converges slowly

5. **Not Bootstrapping**
   - Advantage: unbiased
   - Disadvantage: higher variance than bootstrapping methods
   - Can't leverage value function estimates

### When to Use Monte Carlo

**Use MC when**:
- ✅ No model available
- ✅ Episodes are short
- ✅ Terminal states clearly defined
- ✅ Can run many episodes
- ✅ Need unbiased estimates

**Avoid MC when**:
- ❌ Continuing (non-episodic) task
- ❌ Episodes are very long
- ❌ High variance is problematic
- ❌ Need online learning
- ❌ Sample efficiency critical

### Bridging to Temporal Difference Learning

Monte Carlo methods are a stepping stone to **Temporal Difference (TD) learning** (Lesson 4), which:
- Combines MC sampling with DP bootstrapping
- Works for continuing tasks
- Updates online during episodes
- Often lower variance than MC
- Foundation for Q-learning and SARSA

**Key insight**: MC uses **full returns** (unbiased, high variance), TD uses **estimated returns** (biased, low variance).

---

## Summary

In this notebook, you have learned:

1. ✅ **Model-Free RL**: Learning without transition probabilities
2. ✅ **MC Prediction**: Estimating values by averaging returns
3. ✅ **MC Control**: Finding optimal policies with GPI
4. ✅ **Exploration Strategies**: $\epsilon$-greedy and GLIE
5. ✅ **First vs Every-Visit**: Two MC variants with different properties
6. ✅ **Incremental Updates**: Memory-efficient implementation
7. ✅ **On-Policy vs Off-Policy**: Two fundamental learning paradigms
8. ✅ **Theoretical Guarantees**: Convergence proofs and optimality

### Key Equations

**Return**:
$$G_t = \sum_{k=0}^\infty \gamma^k R_{t+k+1}$$

**MC Value Update**:
$$V(S_t) \leftarrow V(S_t) + \alpha[G_t - V(S_t)]$$

**MC Action-Value Update**:
$$Q(S_t, A_t) \leftarrow Q(S_t, A_t) + \alpha[G_t - Q(S_t, A_t)]$$

**$\epsilon$-Greedy Policy**:
$$\pi(a|s) = \begin{cases}
1 - \epsilon + \frac{\epsilon}{|\mathcal{A}|} & a = \arg\max_{a'} Q(s,a') \\
\frac{\epsilon}{|\mathcal{A}|} & \text{otherwise}
\end{cases}$$

### Practical Insights

- MC is the **simplest** model-free method
- Requires **complete episodes** (episodic tasks)
- **High variance** but **unbiased**
- Exploration is **critical** for convergence
- Foundation for understanding TD learning

### Next Steps

**Lesson 3b** will implement:
- Monte Carlo prediction (first-visit and every-visit)
- Monte Carlo control with $\epsilon$-greedy
- Off-policy MC with importance sampling
- Blackjack environment solution
- Convergence analysis and visualization

---

**Lesson 3a Complete!** 🎉

You now understand the theoretical foundations of Monte Carlo methods for model-free reinforcement learning.